# Pitch Model with Quasi-Steady Aerodynamics

---

Hello! Welcome back to **Introduction to Aeroelastic Instabilities with Jupyter**.

In the previous notebook, we built the **heave-only** model. We discovered something reassuring: the aerodynamics of an airfoil moving straight up and down acts purely as a **positive damper**. No matter how fast we fly, an airfoil restricted to pure vertical motion will always damp its oscillations and never become unstable.

But real wings also **twist**. And, as you might suspect, the story for the equivalent rotation in the typical section is very different.

In this notebook, we focus on **pitch-only** motion, where the typical section can only rotate about its elastic axis. This will force us to answer two key questions:

1. When the airfoil rotates, the relative wind direction is **different at every point** along the chord. Which point should we use to define the effective angle of attack?
2. What happens to the **stability** of the system? Is it, like heave, always stable?

The answers will set the stage perfectly for our next notebook, where we finally combine heave and pitch and encounter our first **Flutter instability**.

### Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how pitch-rate rotation creates a spatially varying effective angle of attack along the chord
2. Apply the quarter-chord vortex / three-quarter-chord collocation result from classical thin-airfoil theory to justify evaluating the quasi-steady angle of attack at the three-quarter chord
3. Derive the equation of motion for the pitch-only typical section using moment equilibrium about the elastic axis
4. Identify the aerodynamic damping and stiffness terms in the equation of motion, and explain their physical significance and sign
5. Implement a state-space simulation to compute the time response and eigenvalues of the pitch model across a range of flight speeds
6. Interpret the root locus and V-g/V-f plots to explain the stability behaviour of the pitch model for different elastic axis positions

*Let's get started!*

In [ ]:
# We start by importing necessary libraries
import numpy as np
import matplotlib.pyplot as plt

# Automatically use "inline" for Google Colab or "widget" for interactive plots in Jupyter
import sys
if 'google.colab' in sys.modules:
    %matplotlib inline
else:
    %matplotlib widget

## 1. The Physics: Pitch-Rate Contribution to the Effective Angle of Attack

In this notebook we return to the simplified **pitch** model that we used in Notebook 1 to introduce the aeroelastic polar and torsional divergence. In this model, the typical section is only allowed to rotate about its elastic axis, and we ignore any vertical translation. However, similarly to the heave-only model of last notebook, we treat it as a *dynamic* system. This means meaning that the pitch angle $\theta(t)$, pitch rate $\dot{\theta}(t)$, and pitch acceleration $\ddot{\theta}(t)$ all vary in time. Also in this case we adopt **quasi-steady** aerodynamics, meaning that the aerodynamic forces depend only on the instantaneous angle of attack. The key new question is: how does pitch motion define an effective angle of attack?

---

### 1.1 A Key Difference from Heave

In the heave model, the airfoil translated straight up or down. At any given instant, **every point along the chord had the same vertical velocity** $\dot{h}$. This made the effective angle of attack simple: it was the same everywhere, equal to $-\dot{h}/V$.

Pitch motion is fundamentally different. When the airfoil **rotates** at an angular rate $\dot{\theta}$, each point along the chord acquires a different velocity perpendicular to the chord. A point close to the elastic axis barely moves, while a point far from it moves a lot.

Take a look at the figure below. During a nose-up rotation ($\dot{\theta} > 0$), points **forward** of the elastic axis (toward the leading edge) acquire an **upward** velocity, while points **aft** of the elastic axis (toward the trailing edge) acquire a **downward** velocity. As we know from the heave model, the airfoil sees a relative velocity opposite to its own motion, which creates an effective angle of attack $\alpha_{\dot{\theta}}$. But since the velocity is different at every point, **the effective angle of attack is also different at every point.**

<figure style="width:75%; margin: auto;">
<img src="../figures/03_airfoil_pitch_rate.svg" style="width:100%">
<figcaption style="text-align: center;">

Airfoil in nose-up pitch rotation: the perpendicular velocity $v_t = \dot{\theta}\,r$ is proportional to the distance $r$ from the elastic axis. Points forward of the EA move upward; points aft move downward.

</figcaption>
</figure>

### 1.2 The Perpendicular Velocity at Each Chord Point

Let us compute this velocity precisely. We use the two reference frames shown in the figure:

- The **inertial frame** $\left(\hat{\boldsymbol{i}}_1, \hat{\boldsymbol{i}}_2, \hat{\boldsymbol{i}}_3\right)$ fixed to the ground.

- The **body frame** $\left(\hat{\boldsymbol{b}}_1, \hat{\boldsymbol{b}}_2, \hat{\boldsymbol{b}}_3\right)$ fixed to the airfoil and moving with it.

When the airfoil rotates at rate $\dot{\theta}$, a chord point at position $r$ from the elastic axis in the direction of $\hat{\boldsymbol{b}}_1$ acquires a velocity **perpendicular to the chord** equal to:

$$\boldsymbol{v}_t = \dot{\theta} \hat{\boldsymbol{b}}_2 \times r \hat{\boldsymbol{b}}_1 = - \dot{\theta} r \hat{\boldsymbol{b}}_3.$$

The unit vector $\hat{\boldsymbol{b}}_3$ has components in the inertial frame given by:

$$\hat{\boldsymbol{b}}_3 = \cos\theta \hat{\boldsymbol{i}}_3 + \sin\theta \hat{\boldsymbol{i}}_1.$$

Consequently, the velocity of the chord point $r$ in the inertial frame is:

$$\boldsymbol{v}_t = - \dot{\theta} r \hat{\boldsymbol{b}}_3 = - \dot{\theta} r \left( \cos\theta \hat{\boldsymbol{i}}_3 + \sin\theta \hat{\boldsymbol{i}}_1 \right).$$

To simplify the expression, we use the small angles assumption, which allows us to approximate $\cos\theta \approx 1$ and $\sin\theta \approx 0$. This gives us:

$$\boldsymbol{v}_t \approx - \dot{\theta} r \hat{\boldsymbol{i}}_3.$$

This simplification allows us to calculate the effective angle of attack $\alpha_{\dot{\theta}}$ at each chord point as:

$$\alpha_{\dot{\theta}}(r) = \tan^{-1}\left(-\frac{v_{t}}{V}\right) \approx -\frac{v_{t}}{V} = \frac{\dot{\theta} r}{V}.$$

The effective angle of attack varies **linearly** along the chord, with a zero crossing exactly at the elastic axis. This is a direct consequence of rigid-body kinematics.

Now we face a problem: the lift model of thin-airfoil theory uses a single angle of attack. Instead, here the angle of attack is a different number at every chord point. Which one do we use?

### 1.3 The Three-Quarter Chord: Why That Specific Point?

The short answer to this question is: **the three-quarter chord**. This is not an arbitrary choice. It is the unique point that makes a simple single-vortex aerodynamic model exactly reproduce the correct lift. Let us see why.

In Notebook 1, we introduced the idea that thin airfoil theory obtains the total lift generated by an airfoil by modelling it as a continuous sheet of infinitesimal vortex elements distributed along the chord and by finding the distribution of vorticity $\gamma(x)$ that satisfies the flow boundary conditions. We also discussed the idea that a lumped vortex element placed at the centroid of the continuous vorticity distribution, that is to say at the quarter chord, produces the same total lift and the same pitching moment about any axis as the full distributed sheet.

The strength of this lumped vortex element corresponds to the so-called circulation $\Gamma$, which is the integral of the continuous vorticity distribution. Thin-airfoil theory shows that for a flat plate at angle of attack $\alpha$, this circulation is:

$$\Gamma = \pi c V \alpha.$$

If we want to determine $\Gamma$ starting from the lumped vortex model instead of integrating the distributed vorticity, we need to impose one physical condition. The natural choice is **flow tangency**: at the surface of the airfoil, the flow velocity perpendicular to the chord must be zero. In other words, no flow can pass through a solid surface. We need to enforce this condition at exactly one representative chord station.

A single point vortex of strength $\Gamma$ at the quarter chord induces a downwash velocity $w$ at a downstream distance $d$ given by the Biot–Savart law for a vortex filament:

$$w = -\frac{\Gamma}{2\pi d},$$

where the negative sign indicates that a clockwise (positive-lift) vortex induces a downward velocity aft of it. The flow tangency condition requires this downwash to cancel the normal component of the freestream:

$$w = -V\sin\alpha \approx -V\alpha,$$

where we have used the small angles approximation. Equating the two expressions for $w$ gives us:

$$\Gamma = 2\pi d\, V \alpha.$$

Comparing with the exact result $\Gamma = \pi c V \alpha$, we find out that we need to impose the flow tangency condition at $d = c/2$. Starting from the quarter chord, a distance of $c/2$ downstream lands exactly at the **three-quarter chord**.

> **The quarter-chord / three-quarter-chord duality:** placing the lumped vortex at the quarter chord and enforcing flow tangency at the three-quarter chord is the *unique* combination that exactly reproduces the correct thin-airfoil lift. The two points are inseparable partners.

### 1.4 Putting It Together: The Effective Angle of Attack of the Pitch Model

We now have all the ingredients. For quasi-steady aerodynamics, we evaluate the angle of attack at the **three-quarter chord** point T as shown in the figure below. Note that, from this point on we use the **half-chord** $b = c/2$ instead of the full chord $c$ as the reference length, which is the standard convention in aeroelasticity.

<figure style="width:75%; margin: auto;">
<img src="../figures/03_typical_section_pitch_dof.svg" style="width:100%">
<figcaption style="text-align: center;">

Typical section with pitch degree of freedom only. The elastic axis (EA) is at $eb$ from the midchord (positive toward the trailing edge). The aerodynamic center (AC) is at the quarter chord ($-b/2$ from midchord). The three-quarter chord collocation point (T) is at $+b/2$ from midchord.

</figcaption>
</figure>

From Section 1.2, the perpendicular velocity at T for a positive pitch rate $\dot{\theta}$ is:

$$v_t = - \dot{\theta} r = - \dot{\theta} \left(\frac{b}{2} - eb\right) = \dot{\theta} b\left(e - \frac{1}{2}\right),$$

where, according to our convention, the distance $r$ is positive aft of the elastic axis, in such a way that a positive pitch rate $\dot{\theta}$ produces a negative velocity $v_t$ at T.

Consequently, the effective angle of attack due to pitch rate is:

$$\alpha_{\dot{\theta}} \approx -\frac{v_t}{V} = \frac{\dot{\theta} b\left(\frac{1}{2} - e\right)}{V}.$$

Adding the quasi-static contribution from the pitch angle $\theta$, the **total effective angle of attack** is:

$$\boxed{\alpha = \theta + \frac{b}{V}\left(\dfrac{1}{2} - e\right)\dot{\theta}.}$$

## 2. Equation of Motion

---

We build the dynamic model by applying Newton's second law for rotation (moment equilibrium about the elastic axis). Three moments act on the typical section:

1. **Elastic restoring moment:** $-k_\theta \theta$ (opposes the rotation with rotational stiffness $k_\theta$).
2. **Aerodynamic moment:** Quasi-steady lift $L$ acts at the quarter chord and its moment arm from the AC to the EA is $eb - (-b/2) = b(1/2 + e)$.
3. **Inertial moment:** $I_{\rm EA} \ddot{\theta}$ (resists angular acceleration with mass moment of inertia $I_{\rm EA}$ about the elastic axis).

Newton's second law for rotation gives:

$$M_{\rm a} - k_\theta \theta = I_{\rm EA} \ddot{\theta}$$

where $M_{\rm a}$ is the aerodynamic moment about the elastic axis, which we can compute by multiplying the lift $L$ by its moment arm:

$$M_{\rm a} = L b\left(\frac{1}{2} + e\right).$$

Substituting $L = q c c_{l\alpha} \alpha$ and $\alpha = \theta + \frac{\dot{\theta} b\left(\frac{1}{2} - e\right)}{V}$, and rearranging (using $c = 2b$), we get:

$$M_{\rm a} = 2 q b c_{l\alpha} \left(\theta + \frac{\dot{\theta} b\left(\frac{1}{2} - e\right)}{V}\right) b\left(\frac{1}{2} + e\right).$$

Rewriting so that all terms appear on the left-hand side gives us the equation of motion:

$$I_{\rm EA} \ddot{\theta} + \left(-\frac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{V}\right) \dot{\theta} + \left(k_\theta - q c_{l\alpha} b^2\left(1 + 2e\right)\right) \theta = 0.$$

### STOP and Look! 👀

Similarly to the heave model, we can identify a second-order linear ordinary differential equation, with inertia, damping, and stiffness terms. However, as you can see, these terms look more complex now. Let's compare them to those of the heave model to understand their physical meaning and implications for stability.

| | Heave model | Pitch model |
|---|---|---|
| Inertia term | $m\ddot{h}$ | $I_{\rm EA}\ddot{\theta}$ |
| Damping term | $\dfrac{q c c_{l\alpha}}{V}$ | $-\frac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{V}$ |
| Stiffness term | $k_h$ | $k_\theta - q c_{l\alpha} b^2\left(1 + 2e\right)$ |

In the heave model, the aerodynamic damping was always positive, meaning that the aerodynamics removed energy from the system, making it always stable. **In the pitch model, the sign of the aerodynamic damping depends on the position of the elastic axis $e$**, and it becomes negative if $\left(1/2 - 2e^2\right) > 0$. This means that the system is intrinsically unstable when $\left|e\right| < 1/2$, that is to say when the elastic axis is located between the quarter and three-quarter chord. Conversely, when the elastic axis is located ahead of the quarter chord or behind the three-quarter chord, the system is intrinsically stable. Why does this happen in our pitch model?

> If $\left|e\right| < 1/2$ (EA between the quarter and three-quarter chords), two effects act simultaneously when the airfoil pitches **nose-up** $\left(\dot{\theta} > 0\right)$: (1) the lift acts *forward* of the elastic axis, producing a nose-up moment; (2) the three-quarter chord moves *downward*, increasing the effective angle of attack and thus the lift. Both effects reinforce each other in the **same direction** as the rotation rate $\dot{\theta}$. In other words, this is positive feedback: the aerodynamics induces more motion instead of opposing it and the system is dynamically unstable.

> If $e<-1/2$ (EA forward of the quarter chord), the lift acts *behind* the elastic axis, so any pitch-rate increase in lift produces a restoring (nose-down) moment. The aerodynamics actively **opposes** the rotation, and the system is always dynamically stable, with no divergence either, as we saw in Notebook 1.

> If $e>1/2$ (EA aft of the three-quarter chord), the lift force acts *ahead* of the elastic axis, but the three-quarter chord moves *upward*, which produces a decrease in the effective angle of attack and consequently in the lift. The pitch rate thus reduces the aerodynamic moment rather than amplifying it, resulting in a **net stabilising effect**, making the system dynamically stable.

Analogously to the heave model, we can also observe that the magnitude of the damping term is proportional to the density $\rho$ and the flight speed $V$, which means that the system becomes more stable or unstable as we fly faster and at lower altitudes.

As for the stiffness term, we can see that it is the sum of the elastic stiffness $k_\theta$ and of the aerodynamic stiffness $- q c_{l\alpha} b^2\left(1 + 2e\right)$. This sum should look familiar to you if you remember the torsional divergence model from Notebook 1. In fact, the pitch model is a dynamic extension of the torsional divergence model, where we have added inertia and damping terms to the equation of motion. If we set the sum of the stiffness terms equal to zero, we can obtain a similar expression for the divergence dynamic pressure as in the torsional divergence model:

$$k_\theta - q c_{l\alpha} b^2\left(1 + 2e\right) = 0 \quad \Longrightarrow \quad q_D = \frac{k_\theta}{b^2\left(1 + 2e\right) c_{l\alpha}},$$

where this time we have used the half-chord $b$ as reference length and a different definition of the location of the elastic axis. This expression tells us that the pitch model can experience static instability in addition to dynamic instability. Analogously to what we saw in Notebook 1, static instability tends to occur earlier (lower flight speed) as the elastic axis moves towards the trailing edge (larger moment arm of the lift force), and it can be completely avoided if the elastic axis is located ahead of the quarter chord $\left(e < -1/2\right)$.

Let's now translate the equation of motion into a state-space model and compute the time response and eigenvalues across a range of flight speeds.

## 3. State-Space Representation and Time Domain Simulation

---

We follow the same procedure as Notebook 2. We isolate the highest derivative:

$$I_{\rm EA} \ddot{\theta} = \frac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{V}\,\dot{\theta} + \left(q c_{l\alpha} b^2\left(1 + 2e\right) - k_\theta\right)\theta,$$

and we define the state vector $\boldsymbol{x} = [\theta, \dot{\theta}]^\intercal$, so that we can write:

$$\dot{\boldsymbol{x}} = \boldsymbol{A}\boldsymbol{x}$$

$$\boldsymbol{A} = \begin{bmatrix} 0 & 1 \\ \dfrac{q c_{l\alpha} b^2\left(1 + 2e\right) - k_\theta}{I_{\rm EA}} & \dfrac{q c_{l\alpha} b^3\left(\frac{1}{2} - 2e^2\right)}{I_{\rm EA} V} \end{bmatrix}.$$

Notice that entry $A_{22}$ is positive when $|e| < 1/2$: a positive pitch rate $\dot{\theta}$ directly accelerates $\ddot{\theta}$ in the same direction, which is the algebraic signature of negative aerodynamic damping.

Let's define the system parameters and the function that builds $\boldsymbol{A}$. In this notebook, the pitch model has more parameters than the heave model, and, more importantly, we will need to analyze the same system for three different elastic axis positions. Instead of using a separate keyword argument for each parameter (as we did in Notebook 2), we will introduce a Python feature that lets us group related parameters into a single, clean container: the `dataclass`.

### Organizing model parameters with Python `dataclass`

In Notebook 2 we listed every parameter as a keyword argument of `build_system_matrix`. That works well when there are few parameters, but the pitch model already needs five parameters, and as we increase the complexity of the typical section models that we consider, we will need even more parameters. Long argument lists become hard to read and easy to get wrong.

Python's `dataclass` (available since Python 3.7, from the `dataclasses` module in the standard library) solves this by letting us group related parameters into a single object. Think of it as a lightweight container: we declare the fields (with names, types, and default values), and Python automatically generates the constructor and a readable string representation for us.

**How it works:**

```python
from dataclasses import dataclass

@dataclass
class PitchParams:
    I_ea: float = 100.0    # inertia [kg*m^2/m]
    k_theta: float = 1e4      # stiffness [N*m/rad/m]
    # ... more fields ...
```

The `@dataclass` line above the class definition is called a **decorator** and it tells Python to automatically add useful features to the class (like a constructor that accepts each field as a keyword argument, and a `__repr__` method that prints the object nicely).

**How to use it:**

```python
# Create an instance with all default values
params = PitchParams()

# Create an instance where only the elastic axis position differs
params_custom = PitchParams(e=0.3)

# Access a field
print(params.k_theta)   # → 10000.0
```

**Why we use it here:** In this notebook we need to analyze the same model for three different elastic axis positions. Instead of calling the function `build_system_matrix` with the specific input argument for the elastic axis position each time, we create separate parameter objects. Each one is self-contained, and `build_system_matrix` receives everything it needs as explicit input.

As you might have noticed in the code snippet above, also with the `dataclass` approach we can define default values for the parameters, corresponding to the baseline configuration of our pitch model:

- Mass moment of inertia $I_{\rm EA} = 100.0$ kgm<sup>2</sup>
- Stiffness $k_\theta = 10^4$ Nm/rad
- Chord $c$ = 1.6 m
- Nondimensional elastic axis position $e = 0.0$ (at mid-chord)
- Lift curve slope $c_{l\alpha} = 2\pi$ 1/rad (thin airfoil theory)

Note that air density $\rho$ is **not** included in the dataclass. Since it is a property of the flow (set by the altitude), not of the typical section itself, we keep it as a separate input argument of `build_system_matrix`, with default value equal to the sea level density $\rho = 1.225$ kg/m<sup>3</sup>.

In [ ]:
from dataclasses import dataclass


@dataclass
class PitchParams:
    """Physical parameters for the 1-DOF pitch aeroelastic model."""

    I_ea: float = 100.0  # mass moment of inertia [kg*m^2]
    k_theta: float = 1e4  # torsional stiffness [N*m/rad]
    c: float = 1.6  # chord length [m]
    e: float = (
        0.0  # elastic axis position (fraction of chord, 0=leading edge, 0.5=mid-chord, 1=trailing edge)
    )
    cl_alpha: float = 2 * np.pi  # lift curve slope [1/rad]


def build_system_matrix(params, velocity, rho=1.225):
    """
    Constructs the system matrix A for the pitch-only typical section model.

    Parameters
    ----------
    params : PitchParams
        Physical parameters of the typical section.
    velocity : float
        Freestream speed [m/s]
    rho : float, optional
        Air density [kg/m^3], by default 1.225 (sea level standard)

    Returns
    -------
    system_matrix : ndarray, shape (2, 2)
        State-space matrix A such that d/dt [theta, theta_dot] = A @ [theta, theta_dot]
    """
    # Calculate semichord
    b = params.c / 2

    # Calculate dynamic pressure from flow conditions
    q = 0.5 * rho * velocity**2

    # Row 1: [0, 1]   (definition: d(theta)/dt = theta_dot)
    # Row 2: [A21, A22]
    #   A21 = (aero stiffness - structural stiffness) / I_ea
    #   A22 = (aero damping numerator) / (I_ea * V)  [positive = negative physical damping]
    A21 = (
        q * params.cl_alpha * b**2 * (1 + 2 * params.e) - params.k_theta
    ) / params.I_ea
    A22 = (
        q
        * params.cl_alpha
        * b**3
        * (1 / 2 - 2 * params.e**2)
        / (params.I_ea * velocity)
    )

    # Assemble the system matrix
    system_matrix = np.array([[0.0, 1.0], [A21, A22]])

    # Return the system matrix
    return system_matrix

Let's now see the pitch model in action by solving the equation of motion in the time domain. Analogously to Notebook 2, we will use the `odeint` function from the `integrate` module of the `scipy` library. To use `odeint`, we first need to define a function that computes the time derivative of the state vector, $\dot{\boldsymbol{x}}$, given the current state $\boldsymbol{x}$ and the system matrix $\boldsymbol{A}$.

In [ ]:
from scipy.integrate import odeint


def state_space_model(x, time_array, system_matrix):
    """
    Returns the time derivative of the state vector for the pitch-only model.

    Parameters
    ----------
    x : array [theta, theta_dot]
    time_array : float  (required by odeint but unused for LTI systems)
    system_matrix : ndarray  (2x2 system matrix A)

    Returns
    -------
    dxdt : array  (time derivative of state vector)
    """
    dxdt = system_matrix @ x
    return dxdt

Now we are going to simulate the response to a small initial disturbance: the airfoil starts at $\theta\left(t=0\right) = 2$ degrees with $\dot{\theta}\left(t=0\right) = 0$ rad/s, and we watch what happens. We start by considering an elastic axis position of $e = -0.6$, so just **forward of the quarter chord**, where we expect the system to always be dynamically stable. Analogously to Notebook 2, we simulate the response at three different flight speeds: 40 m/s, 80 m/s, and 160 m/s. We will plot the time response of $\theta(t)$ for each flight speed. 

In [ ]:
# Define simulation parameters
t = np.linspace(0, 15, 1000)  # simulate for 15 seconds with 1000 time points
initial_condition = [
    np.radians(2),
    0.0,
]  # initial condition: theta = 2 degrees, theta_dot = 0 rad/s

# Create parameter set for EA sligthly ahead of the quarter chord --> e = -0.6
params_ea_ahead_ac = PitchParams(e=-0.6)

# Define flight speeds to test
flight_speeds_ea_ahead_ac = [40, 80, 160]  # m/s

# Create figure
plt.figure()

# Iterate over speeds
for V in flight_speeds_ea_ahead_ac:

    # Build the system matrix for this speed
    A = build_system_matrix(params_ea_ahead_ac, V)

    # Solve the ODE
    solution_array = odeint(state_space_model, initial_condition, t, args=(A,))

    # Extract 'theta' (the first column of the solution)
    theta_response = np.degrees(solution_array[:, 0])

    # Plot
    plt.plot(t, theta_response, label=f"$V = {V}$ m/s")

# Set axes labels, legend, and grid
plt.xlabel("$t$, s")
plt.ylabel("$\\theta$, deg")
plt.legend()
plt.grid(True)
plt.title(f"Time response of pitch model for $e = {params_ea_ahead_ac.e:.2f}$")
plt.show()

As expected for $e = -0.6$, the **system is stable for all flight speeds**, and the **oscillations are more damped at higher flight speeds**, where the aerodynamic damping is stronger.

A notable difference with the heave model is that the **oscillation frequency changes significantly with flight speed**. Can you explain why this happens? Let's go back at the mass-spring-damper analogy to find the answer.

In the last notebook, we reviewed the general solution of the mass-spring-damper system, and we found that the general solution is characterized by the eigenvalues of the characteristic equation. In the case of underdamped response, these eigenvalues can be written as:

$$
\lambda_{1,2} = -\zeta \omega_0 \pm j \omega_0 \sqrt{1-\zeta^2},
$$

where $\omega_0=\sqrt{k/m}$ is the natural frequency of the undamped system, $\zeta=c/2\left(mk\right)^{1/2}$ is the damping ratio, and $k$, $m$, and $c$ are the stiffness, mass, and damping coefficients, respectively. The oscillation frequency of the damped system is given by the imaginary part of the eigenvalues:

$$\omega = \omega_0 \sqrt{1-\zeta^2}.$$

In the heave model, the stiffness and the mass terms were constant, so the only contribution to the change in oscillation frequency with flight speed came from the damping ratio $\zeta$, which typically has a small value for underdamped motion. In the pitch model, instead, the stiffness term includes both an elastic and an aerodynamic component, and the latter changes significantly with flight speed, which leads to a much larger change in the oscillation frequency of the system.

In particular, with $e<-0.5$, the aerodynamic stiffness term $- q c_{l\alpha} b^2\left(1 + 2e\right)$ is positive, and it increases with flight speed. This leads to an increase in the overall stiffness of the system with flight speed, which in turns leads to an increase in the oscillation frequency, as indeed we observe in the time response plots.

This is something very important to keep in mind: **the oscillation frequencies of aeroelastic systems can change significantly with flight speed**, and this has important implications for stability, as we will see in our next notebook. Note how for the typical section we were not able to capture this significant change in oscillation frequency with flight speed by just considering the heave motion, but we needed to consider the effective angle of attack induced by the pitch motion.

Let's now simulate the response of the pitch model for a position of the **elastic axis just behind the quarter chord** $\left(e=-0.4\right)$, where we expect the system to be unstable. As we move the elastic axis from ahead to behind the quarter chord, we also expect the system to experience static instability (divergence) at a certain flight speed. Let's first calculate the divergence speed at sea level, in order to have a reference speed for our simulations. From the expression of the divergence dynamic pressure, we can calculate the divergence speed as:

$$V_D = \sqrt{\frac{2 k_\theta}{\rho c_{l\alpha} b^2\left(1 + 2e\right)}}.$$

In [ ]:
# Create parameter set with EA slightly aft of the aerodynamic center
params_ea_aft_ac = PitchParams(e=-0.4)

# Define density at sea level
rho = 1.225  # kg/m^3

# Calculate divergence speed at sea level
b = params_ea_aft_ac.c / 2
V_D = np.sqrt(
    2 * params_ea_aft_ac.k_theta
    / (rho * params_ea_aft_ac.cl_alpha * b**2 * (1 + 2 * params_ea_aft_ac.e))
)
print(f"Divergence speed at sea level: {V_D:.1f} m/s")

Let's now simulate the response of the pitch model at three different flight speeds based on the calculated divergence speed: $0.5\cdot v_D$, $0.99\cdot v_D$, and $1.01\cdot v_D$.

In [ ]:
# Define flight speeds to test through multipliers of the divergence speed
V_D_multipliers = [0.5, 0.99, 1.01]
flight_speeds_ea_aft_ac = [multiplier * V_D for multiplier in V_D_multipliers]

# Create figure
plt.figure()

# Iterate over speeds
for multiplier, V in zip(V_D_multipliers, flight_speeds_ea_aft_ac):

    # Build the system matrix for this speed
    A = build_system_matrix(params_ea_aft_ac, V)

    # Solve the ODE
    solution_array = odeint(state_space_model, initial_condition, t, args=(A,))

    # Extract 'theta' (the first column of the solution)
    theta_response = np.degrees(solution_array[:, 0])

    # Plot
    plt.plot(
        t, theta_response, label=f"$V = {multiplier:.2f} \\cdot V_D = {V:.1f}$ m/s"
    )

# Set axes labels, legend, and grid
plt.xlabel("$t$, s")
plt.ylabel("$\\theta$, deg")
plt.ylim(-40, 40)  # limit y-axis for better visibility of oscillations
plt.legend()
plt.grid(True)
plt.title(f"Time response of pitch model for $e = {params_ea_aft_ac.e:.2f}$")
plt.show()

What's happening here? We can only see the curve at $1.01\cdot V_D$, with values of $\theta$ that grow exponentially up to an order of $10^{11}$ degrees! This is a clear sign of static instability: the system is diverging in a non-oscillatory way, as we would expect for a flight speed greater or equal to the divergence speed.

To visualize the other two curves at $0.5\cdot V_D$ and $0.99\cdot V_D$, try to add `plt.ylim(-40, 40)` before the `plt.show()` command. What do you see? You should see the curves at $0.5\cdot V_D$ and $0.99\cdot V_D$ diverging in an oscillatory way, and with a noticeably different oscillation frequency. Which curve diverges faster? And which one has a higher oscillation frequency? Can you explain why?

Finally, let's examine the case with the **elastic axis just behind the three-quarter chord** $\left(e=0.6\right)$, where we expect the system to be dynamically stable. However, static instability is still possible in this configuration, so let's first calculate the divergence speed at sea level.

In [ ]:
# Create parameter set with EA slightly aft of the three-quarter chord
params_ea_aft_3q = PitchParams(e=0.6)

# Calculate divergence speed at sea level
b = params_ea_aft_3q.c / 2
V_D = np.sqrt(
    2 * params_ea_aft_3q.k_theta
    / (rho * params_ea_aft_3q.cl_alpha * b**2 * (1 + 2 * params_ea_aft_3q.e))
)
print(f"Divergence speed at sea level: {V_D:.1f} m/s")

The divergence speed is much lower than for $e = -0.4$, because the elastic axis is much further aft of the aerodynamic center, which leads to a much larger moment arm of the lift force. This is the same trend that we observed in Notebook 1: moving the elastic axis aft decreases the divergence speed.

Let's simulate the response at the same three fractional speeds: $0.5\cdot v_D$, $0.99\cdot v_D$, and $1.01\cdot v_D$.

In [ ]:
# Define flight speeds to test through multipliers of the divergence speed
flight_speeds_ea_aft_3q_chord = [multiplier * V_D for multiplier in V_D_multipliers]

# Create figure
plt.figure()

# Iterate over speeds
for multiplier, V in zip(V_D_multipliers, flight_speeds_ea_aft_3q_chord):

    # Build the system matrix for this speed
    A = build_system_matrix(params_ea_aft_3q, V)

    # Solve the ODE
    solution_array = odeint(state_space_model, initial_condition, t, args=(A,))

    # Extract 'theta' (the first column of the solution)
    theta_response = np.degrees(solution_array[:, 0])

    # Plot
    plt.plot(
        t, theta_response, label=f"$V = {multiplier:.2f} \\cdot V_D = {V:.1f}$ m/s"
    )

# Set axes labels, legend, and grid
plt.xlabel("$t$, s")
plt.ylabel("$\\theta$, deg")
plt.ylim(-5, 5)  # limit y-axis for better visibility of oscillations
plt.legend()
plt.grid(True)
plt.title(f"Time response of pitch model for $e = {params_ea_aft_3q.e:.2f}$")
plt.show()

Once again we can only see the curve at $1.01\cdot V_D$ showing an exponential growth of $\theta$, indicating the expected occurrence of divergence. To visualize the other two curves at $0.5\cdot V_D$ and $0.99\cdot V_D$, try to add `plt.ylim(-5, 5)` before the `plt.show()` command. You should see the response at $0.5\cdot V_D$ and $0.99\cdot V_D$ decaying in an oscillatory way, meaning that the system is dynamically stable. Once again you will see an observable difference both in the rate of decay and in the oscillation frequency as we increase the flight speed. Which curve decays faster? And which one has a higher oscillation frequency? Can you explain why?

Something interesting to notice for the case with $e>0.5$ is that, while the system is dynamically stable, it can still experience static instability at a certain flight speed. In some way, the transition from the case with $-0.5<e<0.5$ to the case with $e>0.5$ hints at a typical feature of aircraft wings, which is that while moving the elastic axis further aft of the aerodynamic center can benefit dynamic stability, it will generally be detrimental to static stability. This means that the position of the elastic axis cannot be chosen in such a way to benefit both dynamic and static stability at the same time, and the aircraft designer must find a compromise between the two. However, we should also keep in mind that in reality the position of the elastic axis cannot be chosen freely for aircraft wings, as it is determined by their structural design, which is driven by many different factors.

> **Key insight:** the pitch-only model has *two* distinct instability mechanisms: dynamic (growing oscillations) and static (divergence). While in theory it's' possible to suppress both by placing the elastic axis ahead of the aerodynamic center, as discussed in Notebook 1, this is not a feasible solution for real aircraft wings.

## 4. Eigenvalue Analysis and Root Locus

---

As in Notebook 2, we can understand the dynamics of our system more efficiently by computing the **eigenvalues** of the system matrix $\boldsymbol{A}$ rather than simulating the time response every time.

Let's recall the meaning of the real and imaginary parts of the eigenvalues:

* $\mathbb{Re}(\lambda)$: determines the rate of decay or growth of the oscillations.
* $\mathbb{Im}(\lambda)$: determines the oscillation frequency.

And let's also recall the physical meaning of their different combinations:

| $\mathbb{Re}(\lambda)$ | $\mathbb{Im}(\lambda)$ | Physical meaning |
|---|---|---|
| $< 0$ | $\neq 0$ | Stable, damped oscillation |
| $> 0$ | $\neq 0$ | Unstable, growing oscillation |
| $< 0$ | $= 0$ | Stable, non-oscillatory (overdamped) |
| $> 0$ | $= 0$ | Unstable, non-oscillatory (static divergence) |

Now we calculate the eigenvalues of the pitch model across the range of elastic axis positions $e$ and flight speeds $V$ that we have considered in the time domain simulations. We are going to print the results to the console and assess whether the system is stable or unstable and whether the response is oscillatory or non-oscillatory.

In [ ]:
# Collect the three parameter sets and their corresponding flight speeds
params_list = [params_ea_ahead_ac, params_ea_aft_ac, params_ea_aft_3q]
elastic_axis_positions = [p.e for p in params_list]

# Assemble list of flight speeds for each elastic axis position
# Note: this is a list of lists, where each inner list corresponds to the flight speeds for a specific elastic axis position
flight_speeds_list = [
    flight_speeds_ea_ahead_ac,
    flight_speeds_ea_aft_ac,
    flight_speeds_ea_aft_3q_chord,
]

# Display header
print(
    f"{'e':^5}  {'V, m/s':^6}  {'Eigenvalue 1':^16}  {'Eigenvalue 2':^16}  {'Stability':^9}  {'Response type':^15}"
)
print("-" * 77)

# Iterate over parameter sets and flight speeds
for i, params in enumerate(params_list):
    for V in flight_speeds_list[i]:

        # Build the system matrix for this speed and elastic axis position
        A = build_system_matrix(params, V)

        # Compute eigenvalues
        eigenvalues = np.linalg.eigvals(A)

        # Sort eigenvalues consistently so that lam1 and lam2 always occupy the same
        # column across all rows. The tuple key (-l.imag, -l.real) sorts in descending
        # order: the negative sign turns the default ascending sort into descending.
        # Complex conjugate eigenvalues are sorted positive imaginary part first,
        # real eigenvalues are sorted by descending real part
        lam1, lam2 = sorted(eigenvalues, key=lambda l: (-l.imag, -l.real))

        # Determine stability
        # If real part of all eigenvalues is negative, the system is stable; otherwise, it's unstable
        if np.all(np.real(eigenvalues) < 0):
            stability = "Stable"
        else:
            stability = "Unstable"

        # Determine response type
        # If any eigenvalue has a nonzero imaginary part, the response is oscillatory; otherwise, it's non-oscillatory
        if np.any(np.imag(eigenvalues) != 0):
            response_type = "Oscillatory"
        else:
            response_type = "Non-oscillatory"

        # Format each eigenvalue as a fixed-width string (real: 8 chars, imaginary: 7 chars, j: 1 char = 16 total)
        eig1 = f"{lam1.real:+8.3f}{lam1.imag:+7.3f}j"
        eig2 = f"{lam2.real:+8.3f}{lam2.imag:+7.3f}j"

        # Print results
        print(
            f"{params.e:>5.2f}  {V:>6.1f}  {eig1}  {eig2}  {stability:^9}  {response_type:^15}"
        )

Look at the results. Do they agree with the time domain plots? Do you read negative real parts for stable responses? Do you see zero imaginary parts for non-oscillatory responses? And do the real and imaginary parts change with flight speed as you would expect from the time domain simulations?

As we discussed in Notebook 2, it is also very useful to plot the eigenvalues in the complex plane as a function of flight speed, which is called a **root locus** plot. Let's do that for each of the three locations of the elastic axis that we considered in the time domain simulations.

In [ ]:
# Number of flight speeds to consider for the root locus plot
num_speeds = 200

# Initialize array to store flight speeds and list to store real and imaginary parts of eigenvalues for each elastic axis position
flight_speeds_array = np.empty((len(params_list), num_speeds))  # 2D array 3x200
real_parts_list = []
imag_parts_list = []

# Iterate over parameter sets
for i, params in enumerate(params_list):

    # Pre-allocate arrays for real and imaginary parts (2 eigenvalues per speed)
    real_parts = np.zeros(num_speeds * 2)
    imag_parts = np.zeros(num_speeds * 2)

    # Range of flight speeds: start low, go past the divergence speed
    flight_speeds = np.linspace(1, flight_speeds_list[i][-1] * 1.1, num_speeds)  # m/s

    # Iterate over flight speeds and compute eigenvalues
    for j, V in enumerate(flight_speeds):

        # Build system matrix and compute eigenvalues
        A = build_system_matrix(params, V)
        eigenvalues = np.linalg.eigvals(A)

        # Store real and imaginary parts for both eigenvalues at this speed
        real_parts[j * 2 : (j + 1) * 2] = eigenvalues.real
        imag_parts[j * 2 : (j + 1) * 2] = eigenvalues.imag

    # Append the flight speeds and real and imaginary parts for this elastic axis position to the lists
    flight_speeds_array[i, :] = flight_speeds
    real_parts_list.append(real_parts)
    imag_parts_list.append(imag_parts)

    # Scatter plot: colour encodes flight speed
    plt.figure()
    sc = plt.scatter(
        real_parts,
        imag_parts,
        c=np.repeat(
            flight_speeds, 2
        ),  # repeat each flight speed twice to match the eigenvalue pairs
    )

    # Set axes labels, legend, and grid
    plt.xlabel("$\\mathbb{Re}(\\lambda)$ — exponential growth/decay rate")
    plt.ylabel("$\\mathbb{Im}(\\lambda)$ — oscillation frequency (rad/s)")
    plt.colorbar(sc, label="$V$, m/s")
    plt.grid(True)
    plt.title(f"Root locus for elastic axis position $e = {params.e:.2f}$")
    plt.show()

Let's read the three root locus plots together. First of all, all root loci start near the imaginary axis at very low speed, which is consistent with the fact that aerodynamics is the only source of damping in our system, and with no airflow we have no damping. This is the same behavior that we observed for the heave model in Notebook 2.

As far as the case with $e=-0.6$ is concerned, we can see that the eigenvalues are always in the **left** half of the complex plane, which confirms that the system is inherently stable for all flight speeds. As we increase the flight speed, the magnitude of both the real and imaginary parts of the eigenvalues increases, which is consistent with the fact that the system becomes more damped and has a higher oscillation frequency at higher flight speeds, as we observed in the time domain simulations.

The root locus plot for the case with $e=-0.4$ shows a very different behavior. We can see that at least one eigenvalue is always present in the **right** half of the complex plane, which confirms that the system is inherently unstable. The eigenvalues are initially complex conjugates for low flight speeds, and their imaginary part quickly decreases as speed increases, while their real part increases more slowly. This is consistent with the reduction of the oscillation frequency and the increase of the growth rate that we observed in the time domain simulations. At a certain flight speed, the eigenvalues become purely real, with one positive eigenvalue, which is also consistent with the transition from an oscillatory divergence to a non-oscillatory divergence that we observed earlier.

Finally, for the case with $e=0.6$, we can observe a "complementary" behavior compared to the case with $e=-0.4$. The eigenvalues start as complex conjugates in the **left** half of the complex plane, indicating an initially stable and oscillatory response. As we increase the flight speed, the imaginary part of the eigenvalues decreases quickly, while the real part becomes more negative at a slower rate. At a certain flight speed, the eigenvalues become purely real, with one positive eigenvalue, indicating the transition from an oscillatory stable response to a non-oscillatory unstable response, again consistent with what we observed in the time domain simulations as we exceed the divergence speed.

## 5. V-g and V-f plots

---

Let's now produce the V-g and V-f plots for the pitch model. As we saw in Notebook 2, they essentially compress the root locus into two scalar curves, which are those used routinely by aeroelasticians. We follow exactly the same procedure as in Notebook 2, using the following formulas to compute the oscillation frequency and the damping ratio at each velocity:

- **Frequency:** $\omega = \mathbb{Im}(\lambda)$
- **Damping ratio:** $\zeta = \frac{-\mathbb{Re}(\lambda)}{\sqrt{\mathbb{Re}(\lambda)^2 + \mathbb{Im}(\lambda)^2}}$

Remember that the system is **stable** if $\zeta > 0$, and **unstable** if $\zeta < 0$.

We produce a different pair of V-g and V-f plots for each of the three locations of the elastic axis that we have considered in the time domain simulations.

In [ ]:
# Initialize arrays to store frequency and damping ratio
# Since we have one degree of freedom, there will be one frequency and one damping ratio per velocity
frequencies = np.zeros(num_speeds)  # these will be angular frequencies omega
damping_ratios = np.zeros(num_speeds)

# Iterate over parameter sets
for i, params in enumerate(params_list):

    # We need to iterate over the previously computed real and imaginary parts to calculate frequency and damping ratio
    # However, for how we constructed the arrays, we need to skip every other entry
    for j, (real, imag) in enumerate(
        zip(real_parts_list[i][::2], imag_parts_list[i][::2])
    ):

        # Calculate frequency
        frequencies[j] = np.abs(imag)

        # Calculate Damping Ratio zeta
        damping_ratios[j] = -real / np.sqrt(real**2 + imag**2)

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

    # Plot 1: V-g plot (damping vs velocity)
    ax1.plot(flight_speeds_array[i], damping_ratios)
    ax1.set_ylabel("$\\zeta$")
    ax1.grid(True)

    # Plot 2: V-f plot (frequency vs velocity)
    ax2.plot(flight_speeds_array[i], frequencies)
    ax2.set_ylabel("$\\omega$, rad/s")
    ax2.set_xlabel("$V$, m/s")
    ax2.grid(True)

    # Set title, adjust layout, and show the plots
    plt.suptitle(f"V-g and V-f plots for elastic axis position $e = {params.e:.2f}$")
    plt.tight_layout()
    plt.show()

These three plot pairs summarise the stability picture of the pitch model:

- **$e = -0.6$:** $\zeta > 0$ and increasing (more stable), $\omega$ increasing (stiffer). Unconditionally stable. The $\zeta = 0$ boundary is never crossed.

- **$e = -0.4$:** $\zeta$ starts at a slightly negative value and remains below zero for all speeds (always dynamically unstable). The frequency decreases with speed and it collapses to zero at $V_D$, where $\zeta$ jumps discontinuously. This jump is a plotting artefact of the transition from complex to real eigenvalues at divergence, can you explain why?

- **$e = +0.6$:** $\zeta$ starts positive (dynamically stable) and remains positive up to $V_D$, at which point the frequency collapses to zero and an analogous jump of $\zeta$ occurs.

## 6. Exercise — The Role of the Elastic Axis Position

---

In the sections above we explored three specific values of $e$. Now your task is to look at the picture more systematically by sweeping $e$ continuously from $-1$ to $+1$ at different fixed flight speeds and tracking the eigenvalues. You must fill the code cell below, produce a root locus coloured by $e$, and then interpret the results.

Since `build_system_matrix` takes a `PitchParams` object, you can create a new parameter set for each value of $e$ inside your loop:

```python
V = 40.0  # m/s
for e_val in np.linspace(-1, 1, 50):
    params = PitchParams(e=e_val)
    A = build_system_matrix(params, V)
    eigenvalues = np.linalg.eigvals(A)
    # ... store and plot
```

Start with a flight speed of 40 m/s. Does the root locus cross the imaginary axis at some value of $e$? If so, does it cross it for the values of $e$ that you would expect? Can you identify and characterize different regions of stability (stable vs unstable, oscillatory vs non-oscillatory) as a function of $e$? Can you explain the physical mechanisms behind these different regions?

Successively, increase the flight speed to 60 m/s. Do you see any difference in the crossings of the imaginary axis and in the stability regions compared to the case with 40 m/s? Can you explain these differences based on the physics of the system?

In [ ]:
# Select a flight speed
# ...

# Define elastic axis locations to consider for the root locus plot
# ...

# Pre-allocate arrays for real and imaginary parts (2 eigenvalues per e value)
# ...

# Iterate over elastic axis positions
# ...
# Create PitchParams with the current e value, build system matrix, and compute eigenvalues
# ...
# Store real and imaginary parts for both eigenvalues at this e value
# ...

# Make scatter plot where colour encodes elastic axis position
# ...

## 7. Recap & Conclusion: The Pitch-Only Model in Perspective

---

Congratulations on completing Notebook 3! Let us summarise what we have discovered and reflect on what our model cannot capture.

### Key Takeaways

1. **Spatially varying angle of attack.** Unlike heave, pitch rotation creates a different relative wind direction at every chord point. The corresponding effective angle of attack varies linearly along the chord, and we need to determine a single point where to evaluate it.

2. **The three-quarter chord collocation.** Thin-airfoil theory provides the answer to which point to use: place the bound vortex at the quarter chord and enforce flow tangency at the three-quarter chord. This lumped vortex element exactly reproduces the correct thin-airfoil lift. The effective angle of attack for the pitch model is therefore:
$$\alpha = \theta + \frac{b}{V}\left(\frac{1}{2} - e\right)\dot{\theta}.$$

3. **Sign of aerodynamic damping depends on $e$.** The pitch-rate term modifies the lift, which acts at the quarter chord. Whether the resulting moment reinforces or opposes the rotation depends on the positions of the elastic axis, aerodynamic center, and three-quarter chord:
   - $|e| < 1/2$: **negative damping** (dynamically unstable).
   - $|e| > 1/2$: **positive damping** (dynamically stable).

4. **Two instability mechanisms.** The pitch model can lose stability through dynamic instability (growing oscillations) or static instability (monotonic divergence). These are distinct phenomena and their dominance depends on $e$ and flight speed.

5. **Aeroelastic frequencies shift with speed.** The effective stiffness of the system includes an aerodynamic contribution that changes with the dynamic pressure. Depending on the value of $e$, speed can either increase or decrease the oscillation frequency, something much less evident in a pure translational (heave) model.


### Limitations: What Quasi-Steady Aerodynamics Cannot Capture

Our model rests on a key simplifying assumption — **quasi-steady aerodynamics** — which states that the aerodynamic forces at any instant depend only on the instantaneous angle of attack, as if the flow had always been at that state. This assumption has a precise and important consequence: the sign of the aerodynamic damping coefficient is determined entirely by the geometric parameter $e$. Because the factor $q/V = \frac{1}{2}\rho V$ is always positive and grows with speed, the sign of the damping can never change as airspeed increases. This is why our model produces a system that is either *always* stable or *always* unstable: there is no flight speed at which the aerodynamic damping crosses zero and triggers a transition. In other words, analogously to the heave model of Notebook 2, our pitch model **cannot capture the phenomenon of flutter**.

This limitation stems from the fact that our pitch model **does not represent the complete physical picture.** [In a landmark 1949 paper](https://arc.aiaa.org/doi/full/10.2514/8.11885), Smilg showed that, by using the full unsteady aerodynamic theory of Theodorsen, a pitch-only airfoil *can* experience true flutter, that is to say a transition from stable to unstable at a certain critical airspeed.

The key ingredient that quasi-steady aerodynamics discards is the **memory of the flow**. When a real airfoil oscillates, it continuously sheds vorticity into its wake. This shed wake does not disappear; it keeps inducing a velocity field back onto the airfoil, altering its effective angle of attack in a way that depends on the *history* of the motion, not just the instantaneous state. Crucially, this effect introduces a **phase lag** between the airfoil motion and the resulting aerodynamic force, and the magnitude and sign of this lag depend on airspeed. At a certain critical airspeed, the phase lag is such that the aerodynamic moment does net positive work over a full oscillation cycle, feeding energy into the motion rather than removing it. This is flutter. Quasi-steady aerodynamics, by construction, sees only the instantaneous state and is therefore blind to this mechanism entirely.

> In short: quasi-steady aerodynamics can only evaluate whether the aerodynamic moment opposes or reinforces the *instantaneous* state. Full unsteady aerodynamics allows our analysis to capture whether the aerodynamic moment does net positive work over a *complete oscillation cycle*. This is a fundamentally richer question, and the one that matters for flutter.


### Why We Have Not Seen Flutter Yet — And What It Takes

Looking back across both models built so far, we can now give a precise and complete answer to the natural question: *why have we not encountered flutter in either Notebook 2 or this notebook?*

The answer is **dual**:

1. **We are using quasi-steady aerodynamics.** As shown above, quasi-steady aerodynamics fixes the sign of aerodynamic damping based solely on geometry, preventing any speed-dependent transition. Full unsteady aerodynamics introduces frequency-dependent phase effects that can cause the damping to change sign above a critical airspeed, enabling single-degree-of-freedom flutter even without structural coupling.

2. **We have considered only one degree of freedom at a time.** Even with quasi-steady aerodynamics, a system with *two coupled* degrees of freedom can flutter. The coupling between heave and pitch forces a **phase shift** between the two motions. When pitch and heave oscillate out of phase in a specific way (typically with pitch leading heave by roughly 90 degrees), the maximum aerodynamic lift force aligns with the maximum vertical heaving velocity. Because Power = Force $\times$ Velocity, this phase alignment allows the aerodynamic forces to do **net positive work** on the overall system over a full cycle of oscillation. At a critical airspeed, the energy extracted from the freestream flow exceeds the energy dissipated by the system's natural damping, and the oscillations grow exponentially. This is the classical mechanism of **binary flutter**.

These are two genuinely different routes to the same phenomenon:

| Route | Aerodynamics | Degrees of freedom | Source of net positive work |
|---|---|---|---|
| Smilg (1949) | Full unsteady (Theodorsen) | 1 — pitch only | Wake-induced lag shifts the phase of the aerodynamic moment |
| Classical binary flutter | Quasi-steady | 2 — heave + pitch coupled | Structural coupling shifts the phase between pitch and heave |

Both routes are physically real and important. In this course, we take the **second path**, because it allows us to encounter flutter with the aerodynamic tools we already have, before we extend them to unsteady theory in Notebook 5.


### Coming Up Next: The 2-DOF Heave-Pitch Flutter Model

In **the next notebook**, we will finally couple heave and pitch into the full 2-DOF typical section model. We will:
1. Derive the coupled equations of motion using Lagrange's equations.
2. Assemble the coupled mass, damping, and stiffness matrices.
3. Watch the eigenvalues cross the imaginary axis at a certain **flutter speed**.
4. Produce the classical V-g/V-f plot showing the flutter speed and flutter frequency.

The imaginary axis crossing is coming! 🚀

---
**End of Notebook 3**  
*Save your work and proceed to Notebook 4!*